In [2]:
from pyspark.sql.functions import col, lit
yellow_2024 = spark.read.parquet("Files/taxi_data_2024/*yellow*.parquet")
green_2024  = spark.read.parquet("Files/taxi_data_2024/*green*.parquet")

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 5, Finished, Available, Finished, False)

In [3]:
yellow_2024 = yellow_2024.withColumn("pickup_time", col("tpep_pickup_datetime")) \
                         .withColumn("dropoff_time", col("tpep_dropoff_datetime"))

green_2024 = green_2024.withColumn("pickup_time", col("lpep_pickup_datetime")) \
                       .withColumn("dropoff_time", col("lpep_dropoff_datetime"))

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 6, Finished, Available, Finished, False)

In [4]:
def fix(df):
    return df \
        .withColumn("PULocationID", col("PULocationID").cast("long")) \
        .withColumn("DOLocationID", col("DOLocationID").cast("long")) \
        .withColumn("passenger_count", col("passenger_count").cast("double")) \
        .withColumn("trip_distance", col("trip_distance").cast("double")) \
        .withColumn("fare_amount", col("fare_amount").cast("double")) \
        .withColumn("total_amount", col("total_amount").cast("double"))

yellow_2024 = fix(yellow_2024)
green_2024  = fix(green_2024)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 7, Finished, Available, Finished, False)

In [5]:
yellow_2024 = yellow_2024.withColumn("trip_type", lit(None))
green_2024  = green_2024.withColumn("ehail_fee", lit(None))

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 8, Finished, Available, Finished, False)

In [6]:
cols = [
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "total_amount",
    "trip_distance",
    "passenger_count",
    "pickup_time",
    "dropoff_time"
]

yellow_2024 = yellow_2024.select(cols)
green_2024  = green_2024.select(cols)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 9, Finished, Available, Finished, False)

In [7]:
taxi_2024 = yellow_2024.unionByName(green_2024)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 10, Finished, Available, Finished, False)

In [8]:
files = mssparkutils.fs.ls("Files/taxi_22_23/")
paths = [f.path for f in files]

yellow_dfs = []
green_dfs = []

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 11, Finished, Available, Finished, False)

In [9]:
for p in paths:
    df = spark.read.parquet(p)

    if "tpep_pickup_datetime" in df.columns:
        yellow_dfs.append(df)

    elif "lpep_pickup_datetime" in df.columns:
        green_dfs.append(df)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 12, Finished, Available, Finished, False)

In [10]:
from functools import reduce
yellow_22_23 = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), yellow_dfs)
green_22_23  = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), green_dfs)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 13, Finished, Available, Finished, False)

In [11]:
yellow_22_23 = yellow_22_23.withColumn("pickup_time", col("tpep_pickup_datetime")) \
                           .withColumn("dropoff_time", col("tpep_dropoff_datetime"))

green_22_23 = green_22_23.withColumn("pickup_time", col("lpep_pickup_datetime")) \
                         .withColumn("dropoff_time", col("lpep_dropoff_datetime"))

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 14, Finished, Available, Finished, False)

In [12]:
def fix(df):
    return df \
        .withColumn("PULocationID", col("PULocationID").cast("long")) \
        .withColumn("DOLocationID", col("DOLocationID").cast("long")) \
        .withColumn("passenger_count", col("passenger_count").cast("double")) \
        .withColumn("trip_distance", col("trip_distance").cast("double")) \
        .withColumn("fare_amount", col("fare_amount").cast("double")) \
        .withColumn("total_amount", col("total_amount").cast("double"))

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 15, Finished, Available, Finished, False)

In [13]:
yellow_22_23 = fix(yellow_22_23)
green_22_23  = fix(green_22_23)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 16, Finished, Available, Finished, False)

In [14]:
yellow_22_23 = yellow_22_23.withColumn("ehail_fee", lit(None))
green_22_23  = green_22_23.withColumn("trip_type", lit(None))

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 17, Finished, Available, Finished, False)

In [15]:
cols = [
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "total_amount",
    "trip_distance",
    "passenger_count",
    "pickup_time",
    "dropoff_time"
]

yellow_22_23 = yellow_22_23.select(cols)
green_22_23  = green_22_23.select(cols)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 18, Finished, Available, Finished, False)

In [16]:
taxi_22_23 = yellow_22_23.unionByName(green_22_23)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 19, Finished, Available, Finished, False)

In [17]:
taxi = taxi_22_23.unionByName(taxi_2024)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 20, Finished, Available, Finished, False)

In [18]:
taxi = taxi.fillna({"passenger_count": 0})
taxi = taxi.filter(col("passenger_count") > 0)
taxi = taxi.filter(col("trip_distance") > 0)


StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 21, Finished, Available, Finished, False)

In [19]:
from pyspark.sql.functions import unix_timestamp, col, round
taxi = taxi.withColumn(
    "trip_duration_min",
    (unix_timestamp("dropoff_time") - unix_timestamp("pickup_time")) / 60)
taxi = taxi.filter(col("trip_duration_min") > 0)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 22, Finished, Available, Finished, False)

In [20]:
taxi = taxi.withColumn(
    "trip_duration_min",
    round(col("trip_duration_min")))

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 23, Finished, Available, Finished, False)

In [21]:
taxi = taxi.filter(
    (col("pickup_time") >= "2022-01-01") &
    (col("pickup_time") < "2025-01-01"))

taxi.show()

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 24, Finished, Available, Finished, False)

+------------+------------+-----------+------------+-------------+---------------+-------------------+-------------------+-----------------+
|PULocationID|DOLocationID|fare_amount|total_amount|trip_distance|passenger_count|        pickup_time|       dropoff_time|trip_duration_min|
+------------+------------+-----------+------------+-------------+---------------+-------------------+-------------------+-----------------+
|         142|         236|       14.5|       21.95|          3.8|            2.0|2022-01-01 00:35:40|2022-01-01 00:53:29|             18.0|
|         236|          42|        8.0|        13.3|          2.1|            1.0|2022-01-01 00:33:43|2022-01-01 00:42:07|              8.0|
|         166|         166|        7.5|       10.56|         0.97|            1.0|2022-01-01 00:53:21|2022-01-01 01:02:19|              9.0|
|         114|          68|        8.0|        11.8|         1.09|            1.0|2022-01-01 00:25:21|2022-01-01 00:35:23|             10.0|
|          68

In [22]:
zones = spark.read.csv("Files/taxi_zone_lookup.csv", header=True, inferSchema=True)
zones.show(5)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 25, Finished, Available, Finished, False)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [23]:
zones_pu = zones.selectExpr(
    "LocationID as PULocationID",
    "Borough as PU_Borough",
    "Zone as PU_Zone",
    "service_zone as PU_service_zone")

zones_do = zones.selectExpr(
    "LocationID as DOLocationID",
    "Borough as DO_Borough",
    "Zone as DO_Zone",
    "service_zone as DO_service_zone")

taxi = taxi.join(zones_pu, on="PULocationID", how="left")
taxi = taxi.join(zones_do, on="DOLocationID", how="left")


StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 26, Finished, Available, Finished, False)

In [24]:
from pyspark.sql.functions import unix_timestamp, year, to_date

taxi = taxi.withColumn("year", year("pickup_time")).withColumn("date", to_date("pickup_time"))

taxi.show(5)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 27, Finished, Available, Finished, False)

+------------+------------+-----------+------------+-------------+---------------+-------------------+-------------------+-----------------+----------+--------------------+---------------+----------+--------------------+---------------+----+----------+
|DOLocationID|PULocationID|fare_amount|total_amount|trip_distance|passenger_count|        pickup_time|       dropoff_time|trip_duration_min|PU_Borough|             PU_Zone|PU_service_zone|DO_Borough|             DO_Zone|DO_service_zone|year|      date|
+------------+------------+-----------+------------+-------------+---------------+-------------------+-------------------+-----------------+----------+--------------------+---------------+----------+--------------------+---------------+----+----------+
|         236|         142|       14.5|       21.95|          3.8|            2.0|2022-01-01 00:35:40|2022-01-01 00:53:29|             18.0| Manhattan| Lincoln Square East|    Yellow Zone| Manhattan|Upper East Side N...|    Yellow Zone|2022|

In [25]:
zones = spark.read.csv("Files/zone_coordinates.csv",header=True,inferSchema=True)

zones_pu = zones.selectExpr(
    "LocationID as PULocationID",
    "lat",
    "lon")

taxi = taxi.join(zones_pu, on="PULocationID", how="left")

taxi.show(5)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 28, Finished, Available, Finished, False)

+------------+------------+-----------+------------+-------------+---------------+-------------------+-------------------+-----------------+----------+--------------------+---------------+----------+--------------------+---------------+----+----------+------------------+------------------+
|PULocationID|DOLocationID|fare_amount|total_amount|trip_distance|passenger_count|        pickup_time|       dropoff_time|trip_duration_min|PU_Borough|             PU_Zone|PU_service_zone|DO_Borough|             DO_Zone|DO_service_zone|year|      date|               lat|               lon|
+------------+------------+-----------+------------+-------------+---------------+-------------------+-------------------+-----------------+----------+--------------------+---------------+----------+--------------------+---------------+----+----------+------------------+------------------+
|         142|         236|       14.5|       21.95|          3.8|            2.0|2022-01-01 00:35:40|2022-01-01 00:53:29|     

In [26]:
from pyspark.sql.functions import round, col

taxi = taxi.withColumn("lat_r", round(col("lat"), 2)).withColumn("lon_r", round(col("lon"), 2))

taxi.show(5)


StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 29, Finished, Available, Finished, False)

+------------+------------+-----------+------------+-------------+---------------+-------------------+-------------------+-----------------+----------+--------------------+---------------+----------+--------------------+---------------+----+----------+------------------+------------------+-----+------+
|PULocationID|DOLocationID|fare_amount|total_amount|trip_distance|passenger_count|        pickup_time|       dropoff_time|trip_duration_min|PU_Borough|             PU_Zone|PU_service_zone|DO_Borough|             DO_Zone|DO_service_zone|year|      date|               lat|               lon|lat_r| lon_r|
+------------+------------+-----------+------------+-------------+---------------+-------------------+-------------------+-----------------+----------+--------------------+---------------+----------+--------------------+---------------+----+----------+------------------+------------------+-----+------+
|         142|         236|       14.5|       21.95|          3.8|            2.0|2022-0

In [27]:
from pyspark.sql.functions import col, round, concat, lit

dim_zone = taxi.select(
    "PULocationID",
    "PU_Borough",
    "PU_Zone",
    "PU_service_zone",
    "lat",
    "lon"
).dropDuplicates()

dim_zone = dim_zone.withColumn("lat_r", round(col("lat"), 2)) \
                   .withColumn("lon_r", round(col("lon"), 2))

dim_zone = dim_zone.withColumn(
    "coord",
    concat(
        col("lat_r").cast("string"),
        lit("_"),
        col("lon_r").cast("string")
    )
)
dim_zone = dim_zone.dropDuplicates(["coord"])

dim_zone.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("final_project.my_project.gold.dim_zone")


StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 30, Finished, Available, Finished, False)

In [28]:
taxi.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("final_project.my_project.silver.taxi_enriched")

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 31, Finished, Available, Finished, False)

In [29]:
from pyspark.sql.functions import to_date, avg, round, count, col

aq_table = spark.table("final_project.my_project.bronze.aq_table")
aq2_table = spark.table("final_project.my_project.bronze.aq2_table")

aq = aq_table.unionByName(aq2_table, allowMissingColumns=True)

aq = aq.withColumn("date", to_date("date"))

fact_air = aq.groupBy(
    "date",
    "lat",
    "lon",
    "parameter"
).agg(
    avg("value").alias("avg_pollution")
)

fact_air = fact_air.withColumn("lat_r", round(col("lat"), 2)) \
                   .withColumn("lon_r", round(col("lon"), 2))

fact_air.write.mode("overwrite") \
    .saveAsTable("final_project.my_project.gold.fact_air")

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 32, Finished, Available, Finished, False)

In [30]:
taxi_daily = spark.table("final_project.my_project.gold.fact_taxi_daily")
air = spark.table("final_project.my_project.gold.fact_air")

fact_combined = taxi_daily.join(
    air,
    on=["date", "lat_r", "lon_r"],
    how="left"
)

fact_combined.write.mode("overwrite").saveAsTable("final_project.my_project.gold.fact_combined")

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 33, Finished, Available, Finished, False)

In [31]:
from pyspark.sql.functions import count, sum, avg

taxi_df = spark.read.table("final_project.my_project.silver.taxi_enriched")

fact_taxi_daily = taxi_df.groupBy(
    "date", "lat_r", "lon_r"
).agg(
    count("*").alias("trip_count"),
    sum("total_amount").alias("total_revenue"),
    avg("trip_distance").alias("avg_distance"),
    avg("trip_duration_min").alias("avg_duration")
)

fact_taxi_daily.write.mode("overwrite").saveAsTable("final_project.my_project.gold.fact_taxi_daily")

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 34, Finished, Available, Finished, False)

In [32]:
from pyspark.sql.functions import year, month, dayofmonth, dayofweek

dates = taxi_df.select("date").distinct().orderBy("date")

dim_date = dates \
    .withColumn("year", year("date")) \
    .withColumn("month", month("date")) \
    .withColumn("day", dayofmonth("date")) \
    .withColumn("day_of_week", dayofweek("date"))

dim_date.write.mode("overwrite").saveAsTable("final_project.my_project.gold.dim_date")

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 35, Finished, Available, Finished, False)

In [33]:
fact_combined = fact_taxi_daily.join(
    fact_air,
    on=["date", "lat_r", "lon_r"],
    how="left"
)
fact_combined.write.mode("overwrite").saveAsTable("final_project.my_project.gold.fact_combined")

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 36, Finished, Available, Finished, False)

In [34]:
fx = spark.read.table("final_project.my_project.bronze.fx_table") \
    .withColumn("date", to_date(col("TIME_PERIOD"))) \
    .withColumn("rate", col("OBS_VALUE"))

fact_combined_fx = fact_combined.join(
    fx.select("date", "rate"),
    on="date",
    how="left"
).withColumn(
    "revenue_eur", col("total_revenue") * col("rate")
).withColumn(
    "year", year("date")
)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 37, Finished, Available, Finished, False)

In [35]:
gdp = spark.read.table("final_project.my_project.bronze.gdp") \
    .withColumnRenamed("value", "gdp") \
    .withColumn("year", col("date").cast("int"))

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 38, Finished, Available, Finished, False)

In [36]:
fact_final = fact_combined_fx.join(
    gdp.select("year", "gdp"),
    on="year",
    how="left"
)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 39, Finished, Available, Finished, False)

In [37]:
from pyspark.sql.functions import to_date, col, year, concat, lit, round

dim_zone = spark.read.table("final_project.my_project.gold.dim_zone")

fact_final = fact_final.withColumn(
    "coord",
    concat(
        round(col("lat_r"), 2).cast("string"),
        lit("_"),
        round(col("lon_r"), 2).cast("string")
    )
)

fact_final = fact_final.join(
    dim_zone.select("coord", "PU_Zone", "PU_Borough"),
    on="coord",
    how="left"
)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 40, Finished, Available, Finished, False)

In [38]:
fact_final = fact_final.select(
    "date",
    "year",
    "PU_Zone",
    "PU_Borough",
    "coord",
    "lat_r",
    "lon_r",
    "trip_count",
    "total_revenue",
    "revenue_eur",
    "rate",
    "avg_distance",
    "avg_duration",
    "avg_pollution",
    "parameter",
    "gdp"
)

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 41, Finished, Available, Finished, False)

In [39]:
fact_final.write.mode("overwrite").saveAsTable("final_project.my_project.gold.fact_final")

StatementMeta(, bb2dd1ae-e2f1-4419-97b5-e44e480ab07f, 42, Finished, Available, Finished, False)